In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from keras.layers import (Dropout, Input, Dense, Conv2D, 
                          MaxPooling2D, GlobalAveragePooling2D, 
                          UpSampling2D, Conv2DTranspose, 
                          Reshape, Flatten, Activation, 
                          BatchNormalization,LeakyReLU)
from keras.models import Model, Sequential
from keras.preprocessing import image

from keras.initializers import RandomNormal
from keras.optimizers import Adam
from PIL import Image

width, height, channel = 28, 28, 1

gen_optimizer = Adam(0.0001, 0.5)
disc_optimizer = Adam(0.0002, 0.5)
noise_dim = 100

# Build the generator
def buildImprovedGenerator():
    model = Sequential()

    # First Dense layer to project noise into a high-dimensional space
    model.add(Dense(256 * 7 * 7, input_dim=noise_dim))
    model.add(BatchNormalization(momentum=0.8))
    model.add(Activation("relu"))
    
    # Reshape the output into a 7x7x256 feature map
    model.add(Reshape((7, 7, 256)))

    # Upsample to 14x14
    model.add(UpSampling2D())
    model.add(Conv2D(256, kernel_size=3, padding='same',
                     kernel_initializer=RandomNormal(0, 0.02)))
    model.add(BatchNormalization(momentum=0.8))
    model.add(Activation("relu"))

    # Additional layer to increase depth
    model.add(Conv2D(256, kernel_size=3, padding='same',
                     kernel_initializer=RandomNormal(0, 0.02)))
    model.add(BatchNormalization(momentum=0.8))
    model.add(Activation("relu"))

    # Upsample to 28x28
    model.add(UpSampling2D())
    model.add(Conv2D(128, kernel_size=3, padding='same',
                     kernel_initializer=RandomNormal(0, 0.02)))
    model.add(BatchNormalization(momentum=0.8))
    model.add(Activation("relu"))

    # Additional layer to increase depth
    model.add(Conv2D(128, kernel_size=3, padding='same',
                     kernel_initializer=RandomNormal(0, 0.02)))
    model.add(BatchNormalization(momentum=0.8))
    model.add(Activation("relu"))

    # Final output layer
    model.add(Conv2D(channel, kernel_size=3, padding='same',
                     kernel_initializer=RandomNormal(0, 0.02)))
    model.add(Activation("tanh"))

    return model
improved_generator = buildImprovedGenerator()
improved_generator.summary()

# Build the discriminator
def buildImprovedDiscriminator():
    model = Sequential()

    # First convolutional layer
    model.add(Conv2D(64, kernel_size=3, strides=2, padding='same',
                     kernel_initializer=RandomNormal(0, 0.02),
                     input_shape=(width, height, channel)))
    model.add(LeakyReLU(0.2))
    model.add(Dropout(0.3))  # Adding Dropout for regularization

    # Additional convolutional layers to increase depth
    model.add(Conv2D(128, kernel_size=3, strides=2, padding='same',
                     kernel_initializer=RandomNormal(0, 0.02)))
    model.add(BatchNormalization(momentum=0.8))
    model.add(LeakyReLU(0.2))
    model.add(Dropout(0.3))

    model.add(Conv2D(256, kernel_size=3, strides=2, padding='same',
                     kernel_initializer=RandomNormal(0, 0.02)))
    model.add(BatchNormalization(momentum=0.8))
    model.add(LeakyReLU(0.2))
    model.add(Dropout(0.3))

    model.add(Conv2D(512, kernel_size=3, strides=2, padding='same',
                     kernel_initializer=RandomNormal(0, 0.02)))
    model.add(BatchNormalization(momentum=0.8))
    model.add(LeakyReLU(0.2))
    model.add(Dropout(0.3))

    # Flatten and output layer
    model.add(Flatten())
    model.add(Dense(512))
    model.add(LeakyReLU(0.2))
    model.add(Dropout(0.5))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(loss='binary_crossentropy', optimizer=disc_optimizer)
    return model
improved_discriminator = buildImprovedDiscriminator()
improved_discriminator.summary()

# Build the models
generator = buildImprovedGenerator()
discriminator = buildImprovedDiscriminator()

# Load weights
generator.load_weights('improved_generator_weights_epoch_45.h5')
discriminator.load_weights('improved_discriminator_weights_epoch_45.h5')

# Save the generator model in SavedModel format
generator.save('C:/Users/adven/OneDrive/Documents/school/DevOps/coding/ca2copy/models/generator_model', save_format='tf')

# Define the GAN model
noise = Input(shape=(noise_dim,))
fake_data = generator(noise)
discriminator.trainable = False
output = discriminator(fake_data)
gan = Model(noise, output)
gan.compile(loss='binary_crossentropy', optimizer=gen_optimizer)

# Summary of the GAN model
gan.summary()

Model: "sequential_6"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_9 (Dense)             (None, 12544)             1266944   
                                                                 
 batch_normalization_24 (Ba  (None, 12544)             50176     
 tchNormalization)                                               
                                                                 
 activation_18 (Activation)  (None, 12544)             0         
                                                                 
 reshape_3 (Reshape)         (None, 7, 7, 256)         0         
                                                                 
 up_sampling2d_6 (UpSamplin  (None, 14, 14, 256)       0         
 g2D)                                                            
                                                                 
 conv2d_27 (Conv2D)          (None, 14, 14, 256)      

INFO:tensorflow:Assets written to: C:/Users/adven/OneDrive/Documents/school/DevOps/coding/ca2copy/models/generator_model\assets


INFO:tensorflow:Assets written to: C:/Users/adven/OneDrive/Documents/school/DevOps/coding/ca2copy/models/discriminator_model\assets


INFO:tensorflow:Assets written to: C:/Users/adven/OneDrive/Documents/school/DevOps/coding/ca2copy/models/discriminator_model\assets


Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 100)]             0         
                                                                 
 sequential_8 (Sequential)   (None, 28, 28, 1)         2944129   
                                                                 
 sequential_9 (Sequential)   (None, 1)                 2603009   
                                                                 
Total params: 5547138 (21.16 MB)
Trainable params: 2917505 (11.13 MB)
Non-trainable params: 2629633 (10.03 MB)
_________________________________________________________________
